## Libraries

In [2]:
import random
import numpy as np
import torch
from pathlib import Path
import kagglehub
from dataclasses import dataclass, replace, asdict
import json
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.utils as vutils
from PIL import Image
import matplotlib.pyplot as plt
import time

/home/harsh/Work/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config Cell

In [3]:
@dataclass(frozen= True)
class Config:
    # Reproducibility
    SEED: int = 42

    # Dataset
    DATASET: str = "jessicali9530/celeba-dataset"
    EXPECTED_IMAGES: int = 202_599
    DATA_DIR: Path = (Path.cwd().parent / "Data" / "celeba").resolve()

    # Run / output paths
    RUN_NAME: str = "baseline"
    ARTIFACT_ROOT: Path = (Path.cwd() / "artifacts" / "GAN").resolve()

    # Data Pipeline
    IMAGE_SIZE: int = 64
    CHANNELS: int = 3
    BATCH_SIZE: int = 128
    NUM_WORKERS: int = 4
    CROP_SIZE: int = 148

    def __post_init__(self):
        assert self.IMAGE_SIZE == 64

    @property
    def RUN_DIR(self) -> Path:
        return self.ARTIFACT_ROOT / self.RUN_NAME

    @property
    def CKPT_DIR(self) -> Path:
       return self.RUN_DIR / "checkpoints" 

    @property
    def SAMPLE_DIR(self) -> Path:
        return self.RUN_DIR / "samples"

    @property
    def PLOT_DIR(self) -> Path:
        return self.RUN_DIR / "plots"

    def make_dirs(self):
        for d in (self.RUN_DIR, self.CKPT_DIR,self.SAMPLE_DIR, self.PLOT_DIR):
            d.mkdir(parents = True, exist_ok = True)

    def save(self):
        path = self.RUN_DIR / "config.json"
        path.write_text(json.dumps(asdict(self), indent = 2, default = str))
        return path

cfg = Config()
cfg.make_dirs()
cfg.save()
print(f"Run: {cfg.RUN_NAME} -> {cfg.RUN_DIR}")


Run: baseline -> /home/harsh/Work/Notebooks/Projects/Notebooks/artifacts/GAN/baseline


In [4]:
# Checks For GPU
seed = cfg.SEED
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)   # safe to call even without a GPU

# Falls back to CPU so other people can still run the notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA (built with):", torch.version.cuda)
    print(f"VRAM free/total: {free / 1e9:.1f} / {total / 1e9:.1f} GB")

PyTorch: 2.13.0+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 3060
CUDA (built with): 13.0
VRAM free/total: 11.8 / 12.9 GB


## Data 

In [5]:
# Relative to the notebook (Projects/Notebooks), so it works on any machine
data = cfg.DATA_DIR
image_count = cfg.EXPECTED_IMAGES

data.mkdir(parents=True, exist_ok=True)

def count_jpgs(root):
    return sum(1 for _ in root.rglob("*.jpg"))

existing = count_jpgs(data)

if existing >= image_count:
    print(f"Dataset already present ({existing:,} images). Skipping download.")
else:
    print(f"Found {existing:,} images, expected {image_count:,}. Downloading...")
    kagglehub.dataset_download(
        cfg.DATASET,
        output_dir=str(data),
        force_download=existing > 0,   # re-fetch only if a partial copy exists
    )

# Locate the folder that actually holds the JPEGs (the zip may nest it twice)
first = next(data.rglob("*.jpg"), None)
if first is None:
    raise FileNotFoundError(f"No JPEGs found under {data}. Download failed.")

IMG_DIR = first.parent
n_images = sum(1 for _ in IMG_DIR.glob("*.jpg"))
if n_images < image_count:
    raise RuntimeError(f"Only {n_images:,} images in {IMG_DIR}. Delete {data} and rerun.")

print(f"Ready: {n_images:,} images in {IMG_DIR}")


Dataset already present (202,599 images). Skipping download.
Ready: 202,599 images in /home/harsh/Work/Notebooks/Projects/Data/celeba/img_align_celeba/img_align_celeba


## Pipeline

In [6]:
class CelebADataset(Dataset):
    def __init__(self, img_dir, transform = None):
        self.path = sorted(Path(img_dir).glob("*.jpg"))
        self.transform = transform

    def __len__(self):
        return len(self.path)

    def __getitem__(self, index):
        img = Image.open(self.path[index].convert("RGB"))
        return self.transform(img) if self.transform else img

transform = transforms.Compose([
    transforms.CenterCrop(cfg.CROP_SIZE),
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * cfg.CHANNELS, [0.5] * cfg.CHANNELS),
])

dataset = CelebADataset(IMG_DIR, transform)

loader = DataLoader(
    dataset,
    batch_size= cfg.BATCH_SIZE,
    shuffle= True,
    drop_last= True,
    num_workers= cfg.NUM_WORKERS,
    pin_memory= True,
    persistent_workers= cfg.NUM_WORKERS > 0,
)

print(f"Images: {len(dataset):,} | Batches/epoch: {len(loader):,}")

Images: 202,599 | Batches/epoch: 1,582
